# YOLOv8 Training - Tea Disease Detection
Training YOLOv8m model untuk detect 8 kelas penyakit daun teh.

## CELL 1 — Install Dependencies

In [ ]:
print("Installing required packages...")
import subprocess

# Downgrade PyTorch untuk support P100 GPU
subprocess.run([
    "pip", "install", "-q",
    "torch==2.4.1", "torchvision==0.19.1",
    "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

subprocess.run([
    "pip", "install", "-q",
    "--upgrade-strategy", "only-if-needed",
    "ultralytics", "roboflow", "pyyaml"
], check=True)

import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## CELL 2 — Setup Directories

In [ ]:
from pathlib import Path

base_dir = Path('/kaggle/working/tea_disease_yolo8')
dirs = {
    'dataset': base_dir / 'dataset',
    'models': base_dir / 'models',
    'output': base_dir / 'output',
}

for name, path in dirs.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"✅ {name}: {path}")

## CELL 3 — Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow
import shutil

api_key = "WPrpJznPxK5BhArjmMUg"
workspace = "dyl-hgadx"
project_name = "tehobject"
version = 11

dataset_path = str(dirs['dataset'])

rf = Roboflow(api_key=api_key)
project = rf.workspace(workspace).project(project_name)
dataset_version = project.version(version)

if Path(dataset_path).exists():
    shutil.rmtree(dataset_path)

print("📥 Downloading dataset...")
dataset = dataset_version.download("yolov8", location=dataset_path)
print(f"✅ Dataset downloaded to: {dataset_path}")

# Cari data.yaml dengan cara yang benar
yaml_files = list(Path(dataset_path).rglob('data.yaml'))
if yaml_files:
    yaml_path = yaml_files[0]
    print(f"✅ data.yaml found: {yaml_path}")
else:
    print("❌ data.yaml not found!")
    yaml_path = None

## CELL 4 — Training YOLOv8m (100 epochs)

In [ ]:
from ultralytics import YOLO
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Training on: {device}")

yaml_files = list(dirs['dataset'].rglob('data.yaml'))
data_yaml = str(yaml_files[0])
print(f"📝 Using: {data_yaml}")

model = YOLO('yolov8m.pt')
print("✅ Model loaded: YOLOv8m")

print("\n🎯 Starting training (100 epochs, ~2-4 hours)...\n")

results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=8,
    device=device,
    project=str(dirs['models']),
    name='yolov8_tea_disease',
    patience=20,
    save=True,
    verbose=True,
    optimizer='SGD',
    lr0=0.01,
)

print("\n✅ Training selesai!")
print(f"📁 Model: {dirs['models']}/yolov8_tea_disease/weights/best.pt")

## CELL 5 — Validation

In [ ]:
from ultralytics import YOLO

best_pt_files = list(dirs['models'].rglob('best.pt'))

if best_pt_files:
    best_model_path = str(best_pt_files[-1])
    print(f"✅ Found: {best_model_path}")

    model_eval = YOLO(best_model_path)
    metrics = model_eval.val(data=data_yaml, device=device, imgsz=640)

    print("\n📊 Metrics:")
    print(f"  Precision: {metrics.results_dict.get('metrics/precision(B)', 'N/A')}")
    print(f"  Recall   : {metrics.results_dict.get('metrics/recall(B)', 'N/A')}")
    print(f"  mAP@50   : {metrics.results_dict.get('metrics/mAP50(B)', 'N/A')}")
    print(f"  mAP50-95 : {metrics.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
else:
    print("❌ best.pt not found")

## CELL 6 — Save Models to Working Directory

In [ ]:
import shutil
from pathlib import Path

best_list = list(dirs['models'].rglob('best.pt'))
last_list = list(dirs['models'].rglob('last.pt'))

if best_list:
    dest = Path('/kaggle/working/best.pt')
    shutil.copy(best_list[-1], dest)
    size_mb = dest.stat().st_size / (1024*1024)
    print(f"✅ best.pt saved: {dest} ({size_mb:.1f} MB)")
else:
    print("⚠️  best.pt not found")

if last_list:
    dest = Path('/kaggle/working/last.pt')
    shutil.copy(last_list[-1], dest)
    print(f"✅ last.pt saved: {dest}")

print("\n📥 Download dari tab Output Kaggle!")
print("📝 Test lokal: python3 test_local.py --model best.pt --input video.mp4")